In [1]:
import pickle
import sys
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES
IMBALANCED = False
MEAN_CENTERING = True
emb_space = EMB_SPACES[2] # that's the ProstT5

if IMBALANCED:
    path = f'{IMG_OUTPUT_PATH}/performance/train2-epochs=10/{emb_space[0]}_torch.pkl'
else:
    path = f'{IMG_OUTPUT_PATH}/performance/train,balanced,e=10/{emb_space[0]}_torch.pkl'

with open(path, 'rb') as f:
    results = pickle.load(f)
print(results)

{'test_acc': 0.49925021422450727, 'test_roc_auc_micro': 0.7477836706726834, 'test_roc_auc_macro': 0.7387496101956564, 'test_roc_auc_weighted': 0.7382023306107833, 'test_mcc': 0.24909569791726838, 'test_f1_weighted': 0.4970253613904813, 'test_f1_macro': 0.49739021603287753, 'test_f1_micro': 0.49925021422450727, 'test_auprc_macro': 0.5845695729633705, 'test_auprc_micro': 0.6076289465328546, 'test_auprc_weighted': 0.5837073876788132, 'class_acc_df':              class  accuracy
0          BINDING  0.421287
1  CRYPTIC-BINDING  0.380768
2      NON-BINDING  0.696676, 'cm': array([[1322, 1174,  642],
       [1478, 1180,  441],
       [ 510,  430, 2159]]), 'recall_weighted': 0.49925021422450727, 'recall_macro': 0.49957726037162004, 'recall_micro': 0.49925021422450727}


### MLP performace

In [2]:
import numpy as np
# Recall = diagonal / sum of rows
recall = np.diag(results['cm']) / np.sum(results['cm'], axis=1)

print(f"Per-class recall: {recall}")
binding_recall = recall[0]
cryptic_binding_recall = recall[1]
non_binding_recall = recall[2]


Per-class recall: [0.42128744 0.38076799 0.69667635]


### kNN performance

In [3]:
import pandas as pd
K = 50
if IMBALANCED:
    with open(f'/home/unix/vkrhk/EmmaEmb/EmmaEmb/img/knn-binding-sites/imbalanced/cosine/{emb_space[0]},k={K}{",mean_centered" if MEAN_CENTERING else ""}.csv') as f:
        knn_results = pd.read_csv(f)
else:
    with open(f'/home/unix/vkrhk/EmmaEmb/EmmaEmb/img/knn-binding-sites/cosine/k={K}{",mean_centered" if MEAN_CENTERING else ""}.csv') as f:
        knn_results = pd.read_csv(f)
        knn_results = knn_results[['Unnamed: 0', emb_space[0]]]

for row in knn_results.itertuples():
    if 'CRYPTIC-BINDING' in row[1]:
        cryptic_kNN_score = row[2]
    elif 'NON-BINDING' in row[1]:
        non_binding_kNN_score = row[2]
    else:
        binding_kNN_score = row[2]

print(f"Binding kNN Score: {binding_kNN_score:.4f}")
print(f"Non-binding kNN Score: {non_binding_kNN_score:.4f}")
print(f"Cryptic Binding kNN Score: {cryptic_kNN_score:.4f}")

Binding kNN Score: 0.5021
Non-binding kNN Score: 0.6408
Cryptic Binding kNN Score: 0.4504


### Subplot A

In [4]:
literature_performance_per_binding_class = {
    "Binding": {
        "label_size": 237344 if IMBALANCED else 12970,
        "text_label": "RB",
        "emma_cosine": binding_kNN_score,
        "la": binding_recall,
    },
    "Cryptic-Binding": {
        "label_size": 13263 if IMBALANCED else 12964,
        "text_label": "CB",
        "emma_cosine": cryptic_kNN_score,
        "la": cryptic_binding_recall,
    },
    "Non-Binding": {
        "label_size": 1792157 if IMBALANCED else 12964,
        "text_label": "NB",
        "emma_cosine": non_binding_kNN_score,
        "la": non_binding_recall,
    }

}

In [5]:
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

distance_metric = "cosine"
df = pd.DataFrame(literature_performance_per_binding_class).T.reset_index()
df.columns = ["binding_class",
                "label_size", "text_label", "cosine", "la"]
df['binding_class_name'] = df[
    'binding_class'
    ].str.replace(".", " ")
df['label_size'] = np.sqrt(df['label_size'].to_numpy().astype(float))

df_sorted = df.sort_values(by="label_size", ascending=False).reset_index(
    drop=True
    )
num_classes = df_sorted['binding_class_name'].nunique()
df_sorted['label_size'] = df_sorted['label_size'].astype(float)

fig_1b = px.scatter(
    df_sorted,
    x="la",
    y=distance_metric,
    color="binding_class_name",
    size="label_size",
    text="text_label",
)
fig_1b.update_layout(
    template="plotly_white",
    font={"family": "Arial", "color": "black", "size": 20},
    width=600,
    height=600,
    showlegend=False,
    xaxis=dict(
        showgrid=False,
        linecolor='black',
        linewidth=3,
        title="Accuracy of ESM2 for distinguishing binding site type",
        ticks='outside',           # show ticks outside the axis
        tickwidth=2,               # width of the ticks
        tickcolor='black',         # color of the ticks
        ticklen=6,                 # length of the ticks
        range=[0, 1]
    ),
    yaxis=dict(
        showgrid=False,
        linecolor='black',
        linewidth=3,
        title="Mean KNN feature alignment score",
        ticks='outside',           # show ticks outside the axis
        tickwidth=2,               # width of the ticks
        tickcolor='black',         # color of the ticks
        ticklen=6,                 # length of the ticks 
        range=[0, 1]
    )
)

fig_1b.update_traces(textposition="bottom center")

fig_1b.add_shape(
    type="line",line_width=2, line_dash="dash", line_color="lightgrey",
    x0=0, y0=0, 
    x1=1, y1=1
)

In [6]:
import sys
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis/figs')
import fig_utils

import pandas as pd
df = pd.DataFrame.from_dict(literature_performance_per_binding_class).T
df['label_size'] = pd.to_numeric(df["label_size"])

fig_utils.plot_metric_vs_knn(df, x_col='la', knn_col='emma_cosine', size_col='label_size', label_col='text_label')

In [24]:
data = results['cm'].T

class_names = ["RB", "CB", "NB"]

matrix = pd.DataFrame(
    data, 
    index=class_names,   
    columns=class_names  
)

fig_utils.plot_class_heatmap(matrix, yaxis_title="Ground-truth class", xaxis_title="Predicted class")
# fig_utils.plot_class_heatmap(
#     matrix=matrix,
#     x_from_index=True,
#     colorscale=[(0.0, TEAL_LIGHT), (0.5, TEAL_MID), (1.0, TEAL_DARK)],
#     zmin=0.0,
#     zmax=float(matrix.values.max()),
#     text_annotation_fmt=".2f",
#     text_white_threshold_frac=0.6,
#     use_integer_coords=True,
#     xaxis_title="Predicted class",
#     yaxis_title="Ground-truth class",
#     font_size=16,
#     margin=dict(l=70, r=70, t=20, b=60),
#     output_path=output_dir + "confusion_matrix_recall_norm.png",
#     width=560,
#     height=500,
# )

In [ ]:
matrix = matrix.div(matrix.sum(axis=1), axis=0)
fig_utils.plot_class_heatmap(matrix, yaxis_title="Ground-truth class", xaxis_title="Predicted class", text_annotation_fmt=".2f")


In [28]:
import sys
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import load_dataset_with_all_balanced_classes, mean_center
from emmaemb.vizualisation import get_class_mixing_in_neighborhood
emma = load_dataset_with_all_balanced_classes()

print('Applying mean-centering to embedding spaces...')
mean_center(emma, emb_spaces=['ProstT5'])
emma.calculate_pairwise_distances(
                emb_space='ProstT5',
                metric='cosine',
            )

mixing_counts, class_labels = get_class_mixing_in_neighborhood(
    emma, emb_space='ProstT5', feature='binding_site', k=50, metric='cosine'
)
remap_labels = {
    'BINDING': 'RB',
    'CRYPTIC-BINDING': 'CB',
    'NON-BINDING': 'NB'
}
class_labels = [remap_labels[label] for label in class_labels]
mixing_df = pd.DataFrame(
    mixing_counts, index=class_labels, columns=class_labels
)

mixing_df = mixing_df.div(mixing_df.sum(axis=1), axis=0)
fig_utils.plot_class_heatmap(mixing_df, yaxis_title="Query class", xaxis_title="Neighbor class", text_annotation_fmt=".2f")


38898 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
Applying mean-centering to embedding spaces...
'ProstT5' mean-centred. Cached distances and projections cleared.
Calculating pairwise distances using cosine...
Using GPU for distance calculation.


Computing cosine similarities: 100%|██████████| 389/389 [00:00<00:00, 498.67it/s]
